# Introduction to `pymovements`

- Processing eye movement data
- Open-source python package
- [Code](https://github.com/pymovements/pymovements)
- [Documentation](https://pymovements.readthedocs.io/en/latest/)
- [User Guide](https://pymovements.readthedocs.io/en/latest/user-guide/index.html)
- [Tutorials](https://pymovements.readthedocs.io/en/latest/tutorials/index.html)


## Navigation

0. [Understanding Eye-Tracking Data](./00_understanding-data.ipynb)
1. [Working with Raw Gaze Samples](./01_inspect-raw-samples.ipynb)
2. [Detecting Occumotoric Events](./02_event-detection.ipynb)
3. [Downloading Public Datasets](./03_public-datasets.ipynb)

Bonus: [Plotting Summary](./10_plotting.ipynb)

_Acknowledgement: This presentation builds on work of several `pymovements` authors._

---

# 1. Working with Raw Gaze Samples

In [ ]:
import matplotlib.pyplot as plt

import pymovements as pm
from pymovements.gaze.experiment import Experiment

%config InlineBackend.figure_format = 'svg'

Load gaze data. They are accessible as time-ordered raw samples in `gaze.samples`. 

Example data: [gaze-toy-example.csv](./gaze-toy-example.csv)

Three columns:
- time [ms],
- x [px],
- y [px]

In [ ]:
# Define Experiment
experiment = Experiment(
    screen_width_px=1280,
    screen_height_px=1024,
    screen_width_cm=38,
    screen_height_cm=30.2,
    distance_cm=68,
    origin="upper left",  # orientation of coordinates
    sampling_rate=250.0,  # recodring rate
)

These values are relevant for transformations and calculating measures later.

In [ ]:
# Load example data from csv
gaze = pm.gaze.from_csv(
    "./gaze-toy-example.csv",  # file location
    experiment=experiment,
    time_column="time",  # specify time column
    pixel_columns=["x", "y"],  # specify pixel columns
)

Let's look at the first few samples.

In [ ]:
gaze.samples.head(5)

Each row corresponds to one gaze sample. Timestamps are listed in the ``time`` column, and horizontal and vertical gaze positions in pixel coordinates can be found in the `pixel` column. Depending on the loader and input format, additional channels such as binocular coordinates or quality measures may also be present.

## 1.1 Inspecting Raw Samples with Plots

Visual inspection is an essential first step when working with newly loaded gaze data.
Time-series plots **help reveal problems** like signal loss, noise, blinks, sampling irregularities, or calibration problems before any preprocessing is applied.

Using the `pymovements.plotting.traceplot` function, we can visualize raw gaze samples from a `pymovements.Gaze` object.

In [ ]:
pm.plotting.traceplot(gaze)

The plot shows the 'continuous' trajectory of gaze positions across the stimulus, allowing inspection of spatial gaze behavior.

To get a **clearer impression of the time**:

We can examine each recorded signal separately using `pymovements.plotting.tsplot`. The x-axis represents time, as defined by the gaze sample timestamps. In this example, we plot the `x` and `y` channels. 

In [ ]:
gaze_unnested = gaze.clone()
gaze_unnested.unnest("pixel")
gaze_unnested

In [ ]:
pm.plotting.tsplot(
    gaze_unnested,
    xlabel="time [ms]",
    channels=["pixel_x", "pixel_y"],
    share_y=False,
    line_color="darkblue",
    zero_centered_yaxis=False,
)

You can easily guess the type of data:

A text with 6 lines. The `pixel_x` column hints the fixations.

## 1.2 Data Loss

While one can visually inspect the gaze, it is not an objective measure.

Data Loss is the first data quality metric one should look at. Are there NaNs in the data?

In [ ]:
gaze.measure_samples(
    "data_loss",
    column="pixel",
    sampling_rate=gaze.experiment.sampling_rate,
    # unit="count",
)

[gaze-toy-example.csv](./gaze-toy-example.csv)

In [ ]:
gaze.samples.shape

In [ ]:
9 / 4306

Other data quality measures can be calculated. See later at event measures.

## 1.3 Transforming Raw Samples

Raw pixel coordinates are tied to a specific screen setup and viewing distance. For meaningful interpretation and cross-experiment comparison, gaze samples are often transformed into alternative representations. These transformations operate directly on the raw samples and rely on the experimental metadata defined earlier.

### 1.3.1 `pix2deg()`: From Pixels to Degrees of Visual Angle

Eye trackers typically record gaze positions in screen pixels. While useful for display-based inspection, pixel units depend on screen size and viewing distance and are therefore not comparable across setups. The {py:func}`~pymovements.gaze.transforms.pix2deg` function converts pixel coordinates into degrees of visual angle (dva) using the experiment's screen geometry and viewing distance.

Requirements:
- A pixel-based gaze column must be available (by default named "pixel")
- An {py:class}`~pymovements.Experiment` must be attached to the gaze data, because screen size and distance are needed for the conversion

In [ ]:
gaze.pix2deg()
gaze

Unlike pixels, degrees of visual angle reflect the actual angular displacement of the eye relative to the observer's viewpoint and are therefore **comparable across different screen setups and viewing distances**. In the plot below, the overall shape of the signal remains the same as in pixel space, since only the unit of measurement has changed. However, the scale of the y-axes differs, reflecting the conversion from screen-dependent coordinates to angular units.

In [ ]:
gaze_unnested = gaze.clone()
gaze_unnested.unnest("position")

pm.plotting.tsplot(
    gaze_unnested,
    xlabel="time [ms]",
    channels=["position_x", "position_y"],
    share_y=False,
    line_color="darkblue",
    n_rows=2,
    n_cols=1,
    zero_centered_yaxis=False,
)

### 1.3.2 `pos2vel()`: From Position to Velocity

Many eye-movement measures are derived not from position directly but from its **temporal
derivatives**. Velocity is computed from changes in gaze position over time and is **central to event detection algorithms for saccades and fixations**. In pymovements, velocity is computed explicitly from position data with the {py:func}`~pymovements.gaze.transforms.pos2vel` function, using the sampling rate stored in the eye tracker definition.

In [ ]:
gaze.pos2vel()
gaze

The following plot illustrates velocity, i.e. how quickly the eye moves at each time point. Periods of relative stability (low velocity) typically correspond to fixations, whereas sharp peaks in the signal indicate rapid eye movements such as saccades.

In [ ]:
gaze_unnested = gaze.clone()
gaze_unnested.unnest("velocity")

pm.plotting.tsplot(
    gaze_unnested,
    channels=["velocity_x", "velocity_y"],
    share_y=False,
    line_color="darkblue",
    n_rows=2,
    n_cols=1,
    zero_centered_yaxis=False,
)

plt.show()

For more information on these preprocessing steps, please see the {doc}`Preprocessing Raw Gaze Data <../tutorials/preprocessing-raw-data>`